## 1. Test Agentic RAG

In [12]:
import time
import requests

print("AGENTIC RAG - SCENARIO 1. Out of scope Rejection")
print("="*40)

question = "What is a dog?"
print(f"Question: {question}")
print(f"Excepted: Guardrail should reject (score<60) and explain scope \n")

REQUEST_TIMEOUT = 300
TRUNCATE_ANSWERS = True
TRUNCATE_LENGTH = 200

start_time = time.time()
try:
    response = requests.post(
        "http://localhost:8000/api/v1/ask-agentic",
        json={
            "question": question,
            "top_k": 3,
            "use_hybrid": True,
        },
        timeout=REQUEST_TIMEOUT,
    )
    elapsed = time.time() - start_time
    if response.status_code == 200:
        data = response.json()
        print(f"✓ Agentic RAG ({elapsed:.1f}s)")
        print(f"\nAnswer: {data['answer']}")
        print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
        print(f"\nReasoning steps:")
        for i, step in enumerate(data.get('reasoning_steps', []), 1):
            print(f"  {i}. {step}")

        guardrail_step = next(
            (s for s in data.get('reasoning_steps', []) if 'validated' in s.lower() and 'score' in s.lower()),
            None
        )
        if guardrail_step:
            print(f"\nGuardrail validation: {guardrail_step}")
        if data.get("retrieval_attempts", 0) == 0:
            print("\n✓ Guardrail rejected out of scope question")
        else:
            print("\n✗ Guardrail did not reject out of scope question")
    else:
        print(f"✗ Agentic RAG failed (status code: {response.status_code})")
        print(f"Response:{response.text}")

except Exception as e:
    print(f"✗ Error: {e}")

AGENTIC RAG - SCENARIO 1. Out of scope Rejection
Question: What is a dog?
Excepted: Guardrail should reject (score<60) and explain scope 

✓ Agentic RAG (38.0s)

Answer: content=["I apologize, but I can only help with questions about academic research papers in Computer Science, Artificial Intelligence, and Machine Learning from arXiv.\n\nYour question: 'What is a dog?'\n\nThis appears to be outside my domain of expertise. For questions like this, you might want to try:\n- General-purpose AI assistants for broad knowledge questions\n- Domain-specific resources for topics outside CS/AI/ML\n- Technical documentation if asking about specific software/tools\n\nIf you have a question about AI/ML research papers, I'd be happy to help!"] additional_kwargs={} response_metadata={} id='bab314e9-96f9-4056-b7a5-0b5fb3a3c743' tool_calls=[] invalid_tool_calls=[]

Retrieval attempts: 0

Reasoning steps:
  1. Validated query scope (score: 0/100)
  2. Generated answer from context

Guardrail validation

## 2. Test if the agent correctly retrieve and grades documents for research questions

In [18]:
print("Successful retrieval test")
print("="*40)

question = "What are transformers in machine learning?"
print(f"Question: {question}")
print(f"Excepted: Agent should pass guardrail, retrieve documents and generate answer\n")

start_time = time.time()
try:
    response= requests.post(
        "http://localhost:8000/api/v1/ask-agentic",
        json={
            "query": question,
            "top_k": 3,
            "use_hybrid": True,
            "model": "llama3.2:latest"
        },
        timeout=REQUEST_TIMEOUT
    )

    elapsed = time.time() - start_time
    if response.status_code == 200:
        data = response.json()
        print(f"Agentic RAG ({elapsed:.1f}s)")

        answer = data.get("answer", "")
        print(f"Answer:\n{'-'*50}")
        if TRUNCATE_ANSWERS and len(answer) > 500:
            print(answer[:500] + "...")
            print(f"(truncated, full length: {len(answer)} chars)")

        else:
            print(answer)
        print("-"*50)

        sources = data.get("sources", [])
        print(f"\nSources: {len(sources)} papers")
        if sources:
            for i, source in enumerate(sources, 1):
                if isinstance(source, dict):
                    print(f" {i}. {source.get('title', source.get('id', "Unknown"))}")
                elif isinstance(source, str):
                    print(f" {i}. {source}")
                else:
                    print(f" {i}. {str(source)}")

        print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
        print(f"\nReasoning_steps:")
        for i, step in enumerate(data.get("reasoning_steps", []), 1):
            print(f" {i}. {step}")

        # Check rewritten query field
        if data.get('rewritten_query') is None:
            print(f"\n Query was not rewritten ")
        else:
            print(f"\n Query was rewritten to: {data['rewritten_query']}")

        if data.get('retrieval_attempts', 0) > 0:
            print(f"\n Retrieval attempts: {data['retrieval_attempts']}")
        else:
            print("\n⚠ UNEXPECTED: Agent didn't retrieve for research question")
    else:
        print(f"Failed to retrieve documents. Status code: {response.status_code}")
        print(f"Response: {response.text}")
except Exception as e:
    print(f"Error: {str(e)}")


Successful retrieval test
Question: What are transformers in machine learning?
Excepted: Agent should pass guardrail, retrieve documents and generate answer

Error: HTTPConnectionPool(host='localhost', port=8000): Read timed out. (read timeout=300)


## 3. Query rewritting test

In [ ]:
print("Query rewritting test")
print("=" * 40)

question = "Tell me about AI"
print(f"Question: {question}")
print(f"EExpected: Agent may rewrite query if documents aren't relevant\n")

start_time = time.time()
try:
    response = requests.post(
        "http://localhost:8000/api/v1/ask-agentic",
        json={
            "query": question,
            "top_k": 3,
            "use_hybrid": True,
            "model": "llama3.2:latest"
        },
        timeout=REQUEST_TIMEOUT
    )

    elapsed = time.time() - start_time
    if response.status_code == 200:
        data = response.json()
        answer = data.get("answer", "")
        print(f"✓ Agentic RAG ({elapsed:.1f}s)")
        if TRUNCATE_ANSWERS and len(answer) > 500:
            print(answer[:500] + "...")
            print(f"(truncated, full length: {len(data['answer'])} chars)")
        else:
            print(data['answer'])
        print("-" * 50)
        print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
        print(f"\nReasoning steps:")
        for i, step in enumerate(data.get('reasoning_steps', []), 1):
            print(f"  {i}. {step}")

        # Check for guardrail validation step
        print(f"\nValidating guardrail and rewrite steps:")
        reasoning_steps = data.get('reasoning_steps', [])
        if any("validated" in step.lower() for step in reasoning_steps):
            guardrail_step = next(s for s in reasoning_steps if "validated" in s.lower())
            print(f"✓ Guardrail validated query: {guardrail_step}")
        else:
            print("✗ Guardrail did not validate query")

        if data.get('rewritten_query'):
            print(f"\n Query was rewritten!")
            print(f" Original: {question}")
            print(f" Rewritten: {data['rewritten_query']}")
        elif data.get('retrieval_attempts', 0) > 1:
            print("\n→ Multiple retrieval attempts detected")
            if any("rewritten" in step.lower() for step in reasoning_steps):
                print("  ✓ Rewrite step found in reasoning")
            else:
                print("  ⚠ Multiple attempts but no rewrite info")
        else:
            print("\n→ Query worked on first attempt (no rewrite needed)")
        
        if data.get('retrieval_attempts', 0) > 1:
            print(f"\n✓ Agent performed {data['retrieval_attempts']} retrieval attempts")
    else:
        print(f"✗ Request failed: {response.status_code}")
        print(f"Response: {response.text}")
        
except Exception as e:
    print(f"✗ Error: {e}")


## 4. Multiple out of scope queries

In [ ]:
print("Multiple out of scope queries checking for agentic rag")
print("="*40)

test_queries = [
    ("What is the dog?", "Biology question"),
    ("what's the weather today?", "Weather question"),
    ("Hello, how are you?", "Greeting")
]
print("Testing guardrail rejection with various non-ML/NLP queries:")

for query, description in test_queries:
    print(f"Query: {query}")
    print(f"Type: {description}")
    try:
        response = requests.post(
            "http://localhost:8000/api/v1/ask-agentic",
            json={
                "query": query,
                'top_k': 3,
                "use_hybrid": True
            },
            timeout=30
        )

        if response.status_code == 200:
            data = response.json()
            is_rejected = data['retrieval_attempts'] == 0
            guardrail_step = next(
                (s for s in data['reasoning_steps'] if 'validated' in s.lower() and 'scope' in s.lower()),
                None
            )
            print(f"Result: {'Rejected' if is_rejected else 'Accepted'} (attempts: {data['retrieval_attempts']})")
            if guardrail_step:
                print(f"Guardrail step: {guardrail_step}")
        else:
            print(f"Request failed: {response.status_code}")
    except Exception as e:
        print(f"Error: {str(e)}")

    

## 5. Interactive testing

In [ ]:
def ask_agentic(question: str, show_full_answer: bool = False):
    """Helper function to test agentic RAG.
    
    Args:
        question: The question to ask
        show_full_answer: If True, show full answer regardless of TRUNCATE_ANSWERS setting
    """
    print(f"Question: {question}\n")
    
    start = time.time()
    
    try:
        response = requests.post(
            "http://localhost:8000/api/v1/ask-agentic",
            json={"query": question, "top_k": 3, "use_hybrid": True},
            timeout=REQUEST_TIMEOUT
        )
        
        elapsed = time.time() - start
        
        if response.status_code == 200:
            data = response.json()
            print(f"✓ Response in {elapsed:.1f}s\n")
            
            # Display answer
            answer = data.get('answer', '')
            print(f"Answer:\n{'-'*50}")
            if not show_full_answer and TRUNCATE_ANSWERS and len(answer) > 500:
                print(answer[:500] + "...")
                print(f"(truncated, full length: {len(answer)} chars)")
            else:
                print(answer)
            print('-'*50)
            
            # Display metadata
            print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
            
            # Display sources with validation
            sources = data.get('sources', [])
            print(f"Sources: {len(sources)}")
            if sources:
                for i, source in enumerate(sources[:3], 1):  # Show first 3
                    if isinstance(source, dict):
                        print(f"  {i}. {source.get('title', source.get('id', 'Unknown'))}")
                    elif isinstance(source, str):
                        print(f"  {i}. {source}")
            
            # Display reasoning
            print(f"\nReasoning:")
            for step in data.get('reasoning_steps', []):
                print(f"  • {step}")
        else:
            print(f"✗ Error: {response.status_code}")
            print(response.text)
    except Exception as e:
        print(f"✗ Exception: {e}")

ask_agentic("How does BERT differ from GPT?")

In [ ]:
ask_agentic("What is the capital of France?")
ask_agentic("Explain self-attention mechanisms")